# N6 — Clientes MCP: acessando servidores

* Professor: Julio Cesar dos Reis <a href="mailto:dosreis@unicamp.br">(dosreis@unicamp.br)</a>
* Monitor: Renan dos Santos Morais <a href="mailto:r299211@dac.unicamp.br">(r299211@dac.unicamp.br)</a>
* Monitor: Alejandro Núñez Arroyo <a href="mailto:r299215@dac.unicamp.br">(r299215@dac.unicamp.br)</a>


## 0.1 Pré-requisitos

Para acompanhar este notebook, é esperado que você já tenha:

- entendido os papéis de host, cliente e servidor MCP;
- visto o ciclo de inicialização e a negociação de capacidades;
- noção da diferença entre resources, prompts e tools;
- familiaridade com `async`/`await` em Python;
- noções de subprocessos e variáveis de ambiente.


## 0.2 Objetivos da aula

Ao final deste notebook, você deverá saber:

1. Explicar o que um cliente MCP resolve e o que ele deixa para a aplicação.
2. Escrever um arquivo de configuração `mcpServers` e entender por que cada entrada equivale a um cliente.
3. Conectar-se a um servidor local pelo transporte **stdio** com o SDK oficial.
4. Descobrir tools, resources e prompts de um servidor desconhecido a partir dos schemas.
5. Distinguir, do lado do cliente, erro de execução (`isError`) de erro de protocolo (exceção).
6. Testar um servidor com o transporte **em memória**, sem subir processo.
7. Conectar-se a um servidor **remoto** por Streamable HTTP.
8. Manter vários clientes ativos e rotear chamadas entre servidores.
9. Implementar handlers de **sampling** e **roots**, entendendo que o protocolo é bidirecional.
10. Aplicar allowlist, timeout e aprovação humana no lado do cliente.


## 0.3 Mapa do notebook

1. O que faz um cliente MCP.
2. Configuração: o arquivo `mcpServers`.
3. Cliente 1 — stdio contra um servidor local.
4. Cliente 2 — descoberta e uso das três primitivas.
5. Erros vistos pelo cliente.
6. Cliente 3 — transporte em memória.
7. Cliente 4 — servidor remoto por HTTP.
8. Cliente 5 — vários servidores ao mesmo tempo.
9. Cliente 6 — o cliente também atende requisições.
10. O cliente dentro de um agente.
11. Segurança e governança do lado do cliente.
12. Exercícios, resumo e referências.


## 0.4 Contexto usado nos exemplos

Ao contrário do **Notebook 5**, focado em entender o **Model Context Protocol (MCP)**, neste notebook invertemos o ponto de vista. Aqui somos a **aplicação host**: queremos consumir servidores MCP que já existem sejam nossos, de terceiros e remotos usando o **SDK oficial**.

Vamos escrever seis clientes, cada um mudando um único aspecto em relação ao anterior:

| # | Cliente | O que muda |
|---|---|---|
| 1 | stdio contra um servidor local | o básico: conectar, listar, chamar |
| 2 | descoberta | usar um servidor sem ler documentação |
| 3 | em memória | testar um servidor sem subir processo |
| 4 | HTTP | falar com um servidor remoto |
| 5 | multiplexado | vários servidores ao mesmo tempo |
| 6 | com handlers | atender requisições vindas do servidor |

---


## 0.5 Preparação do ambiente

Precisamos de dois pacotes:

- `mcp`: o SDK oficial, que usaremos para os clientes de baixo nível;
- `fastmcp`: uma camada em cima do SDK, útil para o cliente multiplexado e para o transporte em memória.

Duas seções dependem de rede na primeira execução:

- a **seção 8** baixa um servidor do npm com `npx`, então requer Node.js instalado;
- a **seção 7** conecta a um servidor remoto público.

O resto do notebook roda offline depois da instalação.

Tudo funciona no Google Colab. Se o `npx` não estiver disponível no seu ambiente, a célula abaixo tenta instalar o Node.js automaticamente; se não conseguir, a seção 8 avisa quais servidores ficaram de fora, e o restante continua funcionando.


In [1]:
# Descomente se estiver em um ambiente sem as dependências instaladas.
%pip install -U mcp fastmcp


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.5/766.5 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.0/234.0 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.0/170.0 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.0/273.0 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 21.4 MB/s eta 0:00:00


O servidor `filesystem` (seção 8) roda via `npx`, que vem com o Node.js. A célula abaixo verifica se `npx` já está disponível e, se não estiver, tenta instalar automaticamente — via `conda` ou `apt-get`, dependendo do que existir no seu ambiente. Se nenhum dos dois estiver disponível (ex.: macOS sem Homebrew, Windows), ela imprime o comando manual para você rodar fora do notebook.


In [ ]:
import shutil
import subprocess


def npx_disponivel() -> bool:
    return shutil.which("npx") is not None


def tentar_instalar_nodejs() -> None:
    """Tenta instalar Node.js (que traz o npx) pelo gerenciador disponível."""
    if shutil.which("conda"):
        comando = ["conda", "install", "-y", "-c", "conda-forge", "nodejs"]
    elif shutil.which("apt-get"):
        comando = ["apt-get", "install", "-y", "nodejs", "npm"]
    else:
        comando = None

    if comando is None:
        print("Não encontrei 'conda' nem 'apt-get' automaticamente neste ambiente.")
        print("Instale Node.js manualmente e reinicie o kernel depois:")
        print("  conda:   conda install -c conda-forge nodejs")
        print("  apt:     sudo apt install nodejs npm")
        print("  macOS:   brew install node")
        print("  Windows: https://nodejs.org/")
        return

    print("npx não encontrado. Tentando instalar com:", " ".join(comando))
    try:
        subprocess.run(comando, check=True)
    except Exception as erro:
        print("Instalação automática falhou:", erro)
        print("Rode o comando acima manualmente no terminal e reinicie o kernel.")


if npx_disponivel():
    print("npx já está disponível em:", shutil.which("npx"))
else:
    print("npx não encontrado. Tentando instalar Node.js automaticamente...")
    tentar_instalar_nodejs()
    if npx_disponivel():
        print("npx instalado com sucesso em:", shutil.which("npx"))
    else:
        print("npx segue indisponível.")
        print("Sem problema: a seção 8 detecta isso e avisa quais servidores ficaram de fora.")


In [2]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


def print_json(value: Any) -> None:
    """Imprime objetos Python como JSON formatado."""
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


# Servidores MCP escrevem logs em stderr. No Jupyter isso aparece em vermelho e parece erro, então descartamos. Em uma aplicação real, iria para o log do host.
LOG_SERVIDOR = open(os.devnull, "w")

PASTA = Path("mcp_clientes_aula").resolve()
PASTA.mkdir(exist_ok=True)

# Arquivo de exemplo que os servidores de filesystem vão ler mais adiante.
(PASTA / "resumo.txt").write_text(
    "MCP: um contrato comum entre agentes e sistemas.\n", encoding="utf-8"
)

print("Pasta de trabalho:", PASTA)


Pasta de trabalho: /content/mcp_clientes_aula


---

# 1. O que faz um cliente MCP

Um cliente MCP é a peça do host que fala o protocolo com **um** servidor.

Repare no *um*: a relação é 1:1. Um host que precisa de três servidores mantém três clientes, cada um com sua sessão, seus `id` e suas capacidades negociadas.

O cliente resolve cinco coisas:

| Responsabilidade | O que significa na prática |
|---|---|
| Transporte | subir um subprocesso (stdio) ou abrir uma conexão HTTP |
| Handshake | `initialize`, checar a versão negociada, enviar `notifications/initialized` |
| Correlação | gerar `id`, casar cada response com sua request |
| Descoberta | listar tools, resources e prompts, com seus schemas |
| Tradução de erros | transformar o campo `error` em exceção da linguagem |

E o que ele **não** faz:

- não decide qual tool chamar, quem decide é o modelo, ou a lógica do agente;
- não interpreta o conteúdo devolvido, isso é papel do host;
- não é o agente: MCP é a camada de integração, não o loop de raciocínio.

É por isso que existe um cliente genérico e muitos servidores específicos. O trabalho do cliente é sempre o mesmo, independente do domínio do outro lado.


---

# 2. Configuração: o arquivo `mcpServers`

O host lê um arquivo de configuração que declara os servidores, e levanta um cliente para cada entrada.

**Uma entrada = um cliente = uma conexão.**

O formato tem duas famílias de entrada:

- **servidor local**: `command` + `args`, que o host executa como subprocesso e conversa por stdio;
- **servidor remoto**: `url`, para o qual o host abre uma conexão HTTP.

Vamos declarar três servidores: um nosso, um de terceiros e um remoto.


In [3]:
CONFIG_MCP = {
    "mcpServers": {
        # 1. Servidor local que vamos escrever daqui a pouco (transporte stdio).
        "curso": {
            "command": sys.executable,
            "args": [str(PASTA / "servidor_curso.py")],
        },
        # 2. Servidor de terceiros, publicado no npm. Note que `args` delimita o
        #    escopo: este servidor só enxerga a pasta que passamos.
        "filesystem": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", str(PASTA)],
        },
        # 3. Servidor remoto: sem subprocesso, só uma URL (Streamable HTTP).
        "deepwiki": {
            "url": "https://mcp.deepwiki.com/mcp",
        },
    }
}

print_json(CONFIG_MCP)


{
  "mcpServers": {
    "curso": {
      "command": "/usr/bin/python3",
      "args": [
        "/content/mcp_clientes_aula/servidor_curso.py"
      ]
    },
    "filesystem": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem",
        "/content/mcp_clientes_aula"
      ]
    },
    "deepwiki": {
      "url": "https://mcp.deepwiki.com/mcp"
    }
  }
}


**Segredos vão em `env`, nunca em `args`.** Um servidor que precisa de chave recebe assim:

```json
"github": {
  "command": "python3",
  "args": ["servidor_github.py"],
  "env": { "GITHUB_PERSONAL_ACCESS_TOKEN": "${GITHUB_TOKEN}" }
}
```

**`args` é a fronteira de privilégio.** No servidor de filesystem acima, a pasta passada em `args` é tudo o que ele consegue enxergar. Trocar por `/` daria ao servidor acesso ao disco inteiro. Esse é o menor privilégio na prática.


---

# 3. Cliente 1 — stdio contra um servidor local

Precisamos de um servidor para conversar. Vamos escrever um pequeno servidor do curso com `FastMCP`, em arquivo separado, porque o transporte stdio exige um processo.

Ele expõe as três primitivas, duas tools, um resource e um prompt mais duas tools especiais que só funcionam se o **cliente** colaborar. Voltaremos a elas na seção 9.


In [4]:
SERVIDOR = PASTA / "servidor_curso.py"

SERVIDOR.write_text('''"""Servidor MCP do curso: alvo dos clientes deste notebook."""

import json

from mcp.server.fastmcp import Context, FastMCP
from mcp.types import SamplingMessage, TextContent
from pydantic import BaseModel, Field

# log_level="ERROR" evita que os logs do servidor poluam a saída do notebook.
mcp = FastMCP("CursoMCPServer", log_level="ERROR")

CRONOGRAMA = {
    "A4": {"titulo": "Fundamentos Práticos de LangGraph", "foco": "StateGraph, nós, arestas e ciclos", "tipo": "prático"},
    "A5": {"titulo": "LangGraph com LLMs", "foco": "mensagens, roteamento e revisão", "tipo": "prático"},
    "A6": {"titulo": "Tools e agentes ReAct", "foco": "tools, ToolMessage e loop ReAct", "tipo": "prático"},
    "A7": {"titulo": "Model Context Protocol", "foco": "padronização de integrações", "tipo": "arquitetural"},
}


class Modulo(BaseModel):
    codigo_modulo: str = Field(description="Código do módulo")
    titulo: str = Field(description="Título do módulo")
    foco: str = Field(description="Foco do módulo")
    tipo: str = Field(description="Conceitual, prático ou arquitetural")


class Plano(BaseModel):
    codigo_modulo: str = Field(description="Código do módulo")
    minutos_disponiveis: int = Field(description="Tempo disponível em minutos")
    sugestao: str = Field(description="O que fazer nesse tempo")


@mcp.tool()
def consultar_modulo(codigo_modulo: str) -> Modulo:
    """Consulta título, foco e tipo de um módulo do curso."""
    codigo = codigo_modulo.upper()
    if codigo not in CRONOGRAMA:
        raise ValueError(f"Módulo {codigo} não existe. Disponíveis: {', '.join(CRONOGRAMA)}.")
    return Modulo(codigo_modulo=codigo, **CRONOGRAMA[codigo])


@mcp.tool()
def calcular_plano_estudo(codigo_modulo: str, minutos_disponiveis: int) -> Plano:
    """Sugere um plano de estudo para um módulo, dado o tempo disponível."""
    codigo = codigo_modulo.upper()
    if codigo not in CRONOGRAMA:
        raise ValueError(f"Módulo {codigo} não existe. Disponíveis: {', '.join(CRONOGRAMA)}.")
    if minutos_disponiveis <= 0:
        raise ValueError("minutos_disponiveis deve ser maior que zero.")

    if minutos_disponiveis < 30:
        sugestao = "fazer apenas uma revisão conceitual curta"
    elif minutos_disponiveis < 90:
        sugestao = "estudar os conceitos principais e executar dois exemplos"
    else:
        sugestao = "estudar conceitos, executar exemplos e resolver exercícios"

    return Plano(codigo_modulo=codigo, minutos_disponiveis=minutos_disponiveis, sugestao=sugestao)


@mcp.resource("curso://cronograma")
def cronograma() -> str:
    """Cronograma completo do curso, em JSON."""
    return json.dumps(CRONOGRAMA, ensure_ascii=False, indent=2)


@mcp.prompt()
def tutoria_modulo(codigo_modulo: str) -> str:
    """Prompt de tutoria sobre um módulo do curso."""
    modulo = CRONOGRAMA.get(codigo_modulo.upper())
    if modulo is None:
        raise ValueError(f"Módulo {codigo_modulo} não existe.")
    return (
        "Você é um tutor didático de um curso sobre agentes com LLMs e LangGraph. "
        f"Explique o módulo {codigo_modulo.upper()}: {modulo['titulo']}. "
        f"Foco: {modulo['foco']}."
    )


@mcp.tool()
async def resumir_modulo(codigo_modulo: str, ctx: Context) -> str:
    """Resume um módulo pedindo a geração ao modelo do HOST (sampling)."""
    codigo = codigo_modulo.upper()
    if codigo not in CRONOGRAMA:
        raise ValueError(f"Módulo {codigo} não existe.")

    modulo = CRONOGRAMA[codigo]
    resultado = await ctx.session.create_message(
        messages=[
            SamplingMessage(
                role="user",
                content=TextContent(
                    type="text",
                    text=f"Resuma em uma frase o módulo {codigo}: {modulo['titulo']} (foco: {modulo['foco']}).",
                ),
            )
        ],
        max_tokens=100,
    )
    return resultado.content.text


@mcp.tool()
async def onde_salvar(ctx: Context) -> str:
    """Pergunta ao cliente quais pastas ele autoriza (roots)."""
    roots = await ctx.session.list_roots()
    if not roots.roots:
        return "O cliente não declarou nenhuma raiz; não há onde salvar."
    return "Raízes autorizadas pelo cliente: " + ", ".join(str(r.uri) for r in roots.roots)


if __name__ == "__main__":
    mcp.run()
''', encoding="utf-8")

print("Servidor escrito em:", SERVIDOR)

Servidor escrito em: /content/mcp_clientes_aula/servidor_curso.py


Agora o cliente. São três camadas:

1. `StdioServerParameters` descreve **como iniciar** o servidor é o mesmo conteúdo da entrada `curso` da nossa configuração;
2. `stdio_client` sobe o subprocesso e devolve os dois canais de leitura e escrita;
3. `ClientSession` é o cliente MCP propriamente dito, que fala o protocolo por cima desses canais.

O `initialize()` faz o handshake inteiro: manda a request, confere a versão e envia a notification `notifications/initialized`.


In [5]:
PARAMS_CURSO = StdioServerParameters(
    command=sys.executable,
    args=[str(SERVIDOR)],
    env=dict(os.environ),
)


async def apresentar_servidor() -> None:
    async with stdio_client(PARAMS_CURSO, errlog=LOG_SERVIDOR) as (read, write):
        async with ClientSession(read, write) as session:
            init = await session.initialize()

            print("Servidor:", init.serverInfo.name, init.serverInfo.version)
            print("Versão de protocolo negociada:", init.protocolVersion)
            print("\nCapacidades declaradas pelo servidor:")
            print_json(init.capabilities.model_dump(exclude_none=True))


await apresentar_servidor()


Servidor: CursoMCPServer 1.29.0
Versão de protocolo negociada: 2025-11-25

Capacidades declaradas pelo servidor:
{
  "experimental": {},
  "prompts": {
    "listChanged": false
  },
  "resources": {
    "subscribe": false,
    "listChanged": false
  },
  "tools": {
    "listChanged": false
  }
}


Repare no que voltou: o servidor declarou `tools`, `resources` e `prompts`.

É essa resposta que autoriza o cliente a chamar cada família de métodos. Um servidor que declarasse apenas `tools` recusaria `resources/list` com `-32601`.

E note que a sessão vive dentro do `async with`: ao sair, o subprocesso é encerrado. Um host mantém essas sessões abertas enquanto a aplicação roda.


---

# 4. Cliente 2 — descoberta

Essa é uma das vantagens do MCP: usar um servidor que você nunca viu, sem precisar consultar a documentação.

Tudo o que ele precisa está nos metadados: nome, descrição, `inputSchema` e, quando existe, `outputSchema`. É exatamente essa informação que o host repassa ao modelo para que ele decida o que chamar.

Vamos escrever uma função de descoberta genérica e um pequeno auxiliar para não repetir o `async with` em cada exemplo.

In [6]:
async def com_sessao_curso(funcao):
    """Abre uma sessão com o servidor do curso e entrega a `funcao`."""
    async with stdio_client(PARAMS_CURSO, errlog=LOG_SERVIDOR) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return await funcao(session)


async def descobrir(session: ClientSession) -> None:
    tools = await session.list_tools()
    print(f"TOOLS ({len(tools.tools)}):")
    for tool in tools.tools:
        obrigatorios = ", ".join(tool.inputSchema.get("required", []))
        saida = "tipada" if tool.outputSchema else "texto"
        print(f"  {tool.name}({obrigatorios}) -> saída {saida}")
        print(f"      {tool.description}")

    resources = await session.list_resources()
    print(f"\nRESOURCES ({len(resources.resources)}):")
    for resource in resources.resources:
        print(f"  {resource.uri}  ({resource.mimeType})")

    prompts = await session.list_prompts()
    print(f"\nPROMPTS ({len(prompts.prompts)}):")
    for prompt in prompts.prompts:
        argumentos = ", ".join(a.name for a in (prompt.arguments or []))
        print(f"  {prompt.name}({argumentos})")


await com_sessao_curso(descobrir)


TOOLS (4):
  consultar_modulo(codigo_modulo) -> saída tipada
      Consulta título, foco e tipo de um módulo do curso.
  calcular_plano_estudo(codigo_modulo, minutos_disponiveis) -> saída tipada
      Sugere um plano de estudo para um módulo, dado o tempo disponível.
  resumir_modulo(codigo_modulo) -> saída tipada
      Resume um módulo pedindo a geração ao modelo do HOST (sampling).
  onde_salvar() -> saída tipada
      Pergunta ao cliente quais pastas ele autoriza (roots).

RESOURCES (1):
  curso://cronograma  (text/plain)

PROMPTS (1):
  tutoria_modulo(codigo_modulo)


Com esse catálogo em mãos, o cliente já sabe chamar as três primitivas.

Note a diferença entre elas na resposta: a tool devolve `content` **e** `structuredContent` (porque declara `outputSchema`), o resource devolve `contents` e o prompt devolve `messages`.


In [7]:
async def usar_primitivas(session: ClientSession) -> None:
    resultado = await session.call_tool("consultar_modulo", {"codigo_modulo": "A7"})
    print("TOOL -> isError:", resultado.isError)
    print("  content[0].text:  ", resultado.content[0].text)
    print("  structuredContent:", resultado.structuredContent)

    conteudo = await session.read_resource("curso://cronograma")
    print("\nRESOURCE ->", conteudo.contents[0].mimeType,
          f"({len(conteudo.contents[0].text)} caracteres)")

    prompt = await session.get_prompt("tutoria_modulo", {"codigo_modulo": "A7"})
    mensagem = prompt.messages[0]
    print("\nPROMPT -> role:", mensagem.role)
    print("  ", mensagem.content.text[:90], "...")


await com_sessao_curso(usar_primitivas)


TOOL -> isError: False
  content[0].text:   {
  "codigo_modulo": "A7",
  "titulo": "Model Context Protocol",
  "foco": "padronização de integrações",
  "tipo": "arquitetural"
}
  structuredContent: {'codigo_modulo': 'A7', 'titulo': 'Model Context Protocol', 'foco': 'padronização de integrações', 'tipo': 'arquitetural'}

RESOURCE -> text/plain (507 caracteres)

PROMPT -> role: user
   Você é um tutor didático de um curso sobre agentes com LLMs e LangGraph. Explique o módulo ...


---

# 5. Erros vistos pelo cliente

O notebook anterior mostrou os dois mecanismos de erro do lado do servidor. Do lado do cliente eles chegam de formas diferentes, e o host precisa tratar as duas:

- **erro de execução** volta como um result normal, com `isError=True`. O cliente não levanta exceção: cabe ao host repassar o texto ao modelo, que pode se corrigir;
- **erro de protocolo** vira **exceção** (`McpError`), porque a requisição em si estava errada.

Vamos provocar os dois, e também um caso de fronteira: um resource que não existe.


In [8]:
from mcp.shared.exceptions import McpError


async def demonstrar_erros(session: ClientSession) -> None:
    falha = await session.call_tool("consultar_modulo", {"codigo_modulo": "A99"})
    print("1) módulo inexistente -> isError:", falha.isError)
    print("   ", falha.content[0].text)

    inexistente = await session.call_tool("nao_existe", {})
    print("\n2) tool inexistente -> isError:", inexistente.isError)
    print("   ", inexistente.content[0].text)

    try:
        await session.read_resource("curso://inexistente")
    except McpError as erro:
        print("\n3) resource inexistente -> exceção McpError")
        print("    code:", erro.error.code, "| message:", erro.error.message)


await com_sessao_curso(demonstrar_erros)


1) módulo inexistente -> isError: True
    Error executing tool consultar_modulo: Módulo A99 não existe. Disponíveis: A4, A5, A6, A7.

2) tool inexistente -> isError: True
    Unknown tool: nao_existe

3) resource inexistente -> exceção McpError
    code: 0 | message: Unknown resource: curso://inexistente


Aqui aparecem duas diferenças entre a especificação e a prática.

**Tool inexistente.** A especificação diz que é erro de protocolo `-32602`. Este servidor devolveu `isError=True`. Ou seja: nem sempre o que chega é o que o documento promete.

**Resource inexistente.** A especificação reserva `-32002`, que foi exatamente o que implementamos no N5. Este servidor levantou a exceção com outro código.

A lição é que **um cliente robusto não decide seu fluxo pelo código numérico**. A distinção que importa é estrutural, e essa sempre vale:

- veio um result com `isError=True` → devolva o texto ao modelo e deixe ele tentar de novo;
- veio uma exceção → é falha de integração: registre no log e não insista.


---

# 6. Cliente 3 — transporte em memória

Nem todo cliente precisa de processo. Quando o servidor é seu e está no mesmo programa dá para conectar cliente e servidor por streams em memória.

O código de aplicação é idêntico; some apenas o subprocesso.

Aqui usamos o `Client` da biblioteca `fastmcp`, que aceita um objeto servidor diretamente.


In [9]:
from fastmcp import Client, FastMCP

servidor_de_teste = FastMCP("ServidorDeTeste")


@servidor_de_teste.tool()
def somar(a: int, b: int) -> int:
    """Soma dois números."""
    return a + b


async def testar_em_memoria() -> None:
    async with Client(servidor_de_teste) as client:
        tools = await client.list_tools()
        print("tools:", [tool.name for tool in tools])

        resultado = await client.call_tool("somar", {"a": 2, "b": 3})
        print("content:          ", resultado.content[0].text)
        print("structured_content:", resultado.structured_content)


await testar_em_memoria()


tools: ['somar']
content:           5
structured_content: {'result': 5}


É o transporte que você quer nos testes do seu próprio servidor: sem processos, sem portas, sem espera e ainda assim passando pelo protocolo inteiro, incluindo handshake e serialização.

---

# 7. Cliente 4 — servidor remoto por HTTP

Servidores MCP não precisam rodar na sua máquina. O transporte **Streamable HTTP** conecta o cliente a um serviço na internet.

Vamos usar o **DeepWiki**, um servidor público e sem chave que responde perguntas sobre repositórios do GitHub.

Compare com a seção 3: muda a função de transporte e some o `StdioServerParameters`. `ClientSession` e todo o resto continuam iguais.


In [10]:
from mcp.client.streamable_http import streamablehttp_client

URL_REMOTA = "https://mcp.deepwiki.com/mcp"


async def cliente_remoto(repositorio: str) -> None:
    async with streamablehttp_client(URL_REMOTA) as (read, write, _):
        async with ClientSession(read, write) as session:
            init = await session.initialize()
            print("Servidor remoto:", init.serverInfo.name,
                  "| protocolo:", init.protocolVersion)

            tools = await session.list_tools()
            print("tools:", [tool.name for tool in tools.tools], "\n")

            resultado = await session.call_tool(
                "read_wiki_structure", {"repoName": repositorio}
            )
            print(resultado.content[0].text[:350], "...")


await cliente_remoto("modelcontextprotocol/python-sdk")


Servidor remoto: DeepWiki | protocolo: 2025-11-25
tools: ['ask_question', 'read_wiki_contents', 'read_wiki_structure'] 

Available pages for modelcontextprotocol/python-sdk:

- 1 Overview
  - 1.1 Installation & Dependencies
  - 1.2 Key Concepts & Architecture
- 2 FastMCP / MCPServer Framework
  - 2.1 Creating a FastMCP Server
  - 2.2 Tool System
  - 2.3 Resources & Prompts
  - 2.4 Function Metadata & Schema Generation
  - 2.5 Context Injection & Lifespan Management
  ...


Servidores remotos privados costumam exigir autenticação, normalmente OAuth. O SDK oferece um fluxo pronto para isso, mas ele depende de credenciais e de um navegador.

O que importa reter: **o contrato não muda com o transporte**. O mesmo `list_tools` e o mesmo `call_tool` servem para um subprocesso local e para um serviço remoto.


---

# 8. Cliente 5 — vários servidores ao mesmo tempo

Um host não conversa com um servidor só. Ele lê a configuração e mantém N clientes ativos.

O `Client` do `fastmcp` aceita o dicionário `mcpServers` inteiro e multiplexa tudo: por baixo há um cliente por servidor, mas a aplicação enxerga um catálogo único.

Para evitar colisão de nomes, as tools vêm prefixadas com o nome do servidor: `curso_consultar_modulo`, `filesystem_read_text_file`. Esse prefixo é a chave de roteamento.

> A primeira execução baixa o servidor de filesystem do npm, então pode demorar alguns segundos.

**O stderr do kernel não serve para subprocessos.** Jupyter e Colab substituem `sys.stderr` por um objeto próprio, que não tem descritor de arquivo. Como o cliente multiplexado entrega o `sys.stderr` ao subprocesso do servidor, a conexão falha com `UnsupportedOperation: fileno`. A solução é desviar o stderr durante a conexão, e é por isso que criamos o `LOG_SERVIDOR` lá no começo.


In [11]:
import contextlib

from fastmcp import Client


@contextlib.asynccontextmanager
async def cliente_multiplexado(config: dict = CONFIG_MCP):
    """Abre um cliente para todos os servidores da configuração.

    Dois cuidados que valem para qualquer host rodando dentro de um notebook:

    1. `redirect_stderr` é obrigatório aqui. O cliente multiplexado usa
       `sys.stderr` para o log dos subprocessos, e o stderr do kernel não tem
       descritor de arquivo sem o desvio, nenhum servidor stdio sobe.
    2. Quando um servidor falha, o `fastmcp` apenas o ignora e segue com os
       outros. Isso é útil em produção, mas silencioso demais para aprender:
       a falha só apareceria depois, como "Unknown tool".
    """
    with contextlib.redirect_stderr(LOG_SERVIDOR):
        async with Client(config) as client:
            if len(config["mcpServers"]) > 1:
                nomes = [tool.name for tool in await client.list_tools()]
                faltando = [
                    servidor
                    for servidor in config["mcpServers"]
                    if not any(nome.startswith(servidor + "_") for nome in nomes)
                ]
                if faltando:
                    print("AVISO: não conectaram e serão ignorados:", ", ".join(faltando))

            yield client


async def catalogo_do_host() -> None:
    async with cliente_multiplexado() as client:
        tools = await client.list_tools()

        por_servidor: dict[str, list[str]] = {}
        for tool in tools:
            servidor, _, nome = tool.name.partition("_")
            por_servidor.setdefault(servidor, []).append(nome)

        print(f"{len(tools)} tools em {len(por_servidor)} servidores:\n")
        for servidor, nomes in por_servidor.items():
            print(f"  {servidor:12} {len(nomes):2} tools   ex.: {nomes[:2]}")


await catalogo_do_host()


[08/01/26 15:47:36] INFO     Proxy detected connected client - reusing existing session for all        ]8;id=843355;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=485424;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py#837\837]8;;\
                             requests. This may cause context mixing in concurrent scenarios.                      

[08/01/26 15:47:49] INFO     Proxy detected connected client - reusing existing session for all        ]8;id=779089;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=487479;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py#837\837]8;;\
                             requests. This may cause context mixing in concurrent scenarios.                      

                    INFO     Proxy detected connected client - reusing existing session for all        ]8;id=375155;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=712947;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py#837\837]8;;\
                             requests. This may cause context mixing in concurrent scenarios.                      

21 tools em 3 servidores:

  curso         4 tools   ex.: ['consultar_modulo', 'calcular_plano_estudo']
  filesystem   14 tools   ex.: ['read_file', 'read_text_file']
  deepwiki      3 tools   ex.: ['ask_question', 'read_wiki_contents']


Com o catálogo montado, o host roteia cada chamada para o servidor certo apenas olhando o prefixo sem saber onde cada servidor roda nem por qual transporte.

> **Cuidado com o prefixo.** O `fastmcp` só prefixa quando há **mais de um** servidor na configuração. Se você reduzir a `CONFIG_MCP` a um único servidor, as tools voltam a se chamar `consultar_modulo` e `read_text_file`, e o código abaixo quebra. Não escreva o prefixo na mão: pegue os nomes de `list_tools()`.


In [12]:
async def usar_varios_servidores() -> None:
    async with cliente_multiplexado() as client:
        modulo = await client.call_tool("curso_consultar_modulo", {"codigo_modulo": "A7"})
        print("curso      ->", modulo.structured_content)

        arquivo = await client.call_tool(
            "filesystem_read_text_file", {"path": str(PASTA / "resumo.txt")}
        )
        print("filesystem ->", arquivo.content[0].text.strip())


await usar_varios_servidores()


[08/01/26 15:47:53] INFO     Proxy detected connected client - reusing existing session for all        ]8;id=256568;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=379031;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py#837\837]8;;\
                             requests. This may cause context mixing in concurrent scenarios.                      

[08/01/26 15:47:54] INFO     Proxy detected connected client - reusing existing session for all        ]8;id=576456;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=307213;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py#837\837]8;;\
                             requests. This may cause context mixing in concurrent scenarios.                      

                    INFO     Proxy detected connected client - reusing existing session for all        ]8;id=424313;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=805326;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py#837\837]8;;\
                             requests. This may cause context mixing in concurrent scenarios.                      

curso      -> {'codigo_modulo': 'A7', 'titulo': 'Model Context Protocol', 'foco': 'padronização de integrações', 'tipo': 'arquitetural'}
filesystem -> MCP: um contrato comum entre agentes e sistemas.


Três servidores, dois transportes, um catálogo. É esse o desenho que um host de produção usa e é o mesmo que aparece em Claude Desktop, Cursor ou VS Code quando você adiciona servidores na configuração.

---

# 9. Cliente 6 — o cliente também atende requisições

Até aqui o cliente só perguntou. Mas o MCP é **bidirecional**: o servidor também pode fazer requisições ao cliente.

São três capacidades declaradas pelo lado do cliente no `initialize`:

| Capacidade | O servidor pede | Por que existe |
|---|---|---|
| `sampling` | uma geração ao modelo do host | o servidor faz IA sem ter chave de API |
| `roots` | quais pastas ele pode acessar | o cliente define o escopo |
| `elicitation` | um dado ao usuário, no meio da execução | pedir confirmação ou informação faltante |

`sampling` é a mais interessante. O servidor precisa de uma geração, mas quem tem o modelo e a chave é o host, então o servidor manda `sampling/createMessage` e o **cliente** decide se atende.

A especificação recomenda humano no circuito: o cliente deve poder recusar, e a recusa tem código de erro próprio (`-1`).

Nosso `servidor_curso.py` tem duas tools que dependem disso: `resumir_modulo` (sampling) e `onde_salvar` (roots). Vamos registrar os handlers.


In [13]:
from mcp.shared.context import RequestContext
from mcp.types import (
    CreateMessageRequestParams,
    CreateMessageResult,
    ListRootsResult,
    Root,
    TextContent,
)


async def tratar_sampling(
    context: RequestContext["ClientSession", None],
    params: CreateMessageRequestParams,
) -> CreateMessageResult:
    """O servidor pediu uma geração. Quem decide é o host."""
    pedido = params.messages[0].content.text
    print("  [host] o servidor pediu uma geração:", pedido[:60], "...")

    # Aqui o host chamaria a SUA LLM. Simulamos para não depender de chave.
    # Numa aplicação real, este é o ponto de pedir aprovação ao usuário.
    return CreateMessageResult(
        role="assistant",
        content=TextContent(
            type="text",
            text="MCP padroniza como agentes acessam tools e dados externos.",
        ),
        model="simulado-pelo-host",
    )


async def tratar_roots(
    context: RequestContext["ClientSession", None],
) -> ListRootsResult:
    """O servidor perguntou o que pode acessar. O cliente delimita."""
    print("  [host] o servidor perguntou quais pastas pode acessar")
    return ListRootsResult(roots=[Root(uri=PASTA.as_uri(), name="material do curso")])


async def cliente_com_handlers() -> None:
    async with stdio_client(PARAMS_CURSO, errlog=LOG_SERVIDOR) as (read, write):
        async with ClientSession(
            read,
            write,
            sampling_callback=tratar_sampling,
            list_roots_callback=tratar_roots,
        ) as session:
            await session.initialize()

            resumo = await session.call_tool("resumir_modulo", {"codigo_modulo": "A7"})
            print("  resultado:", resumo.content[0].text, "\n")

            raizes = await session.call_tool("onde_salvar", {})
            print("  resultado:", raizes.content[0].text)


await cliente_com_handlers()


  [host] o servidor pediu uma geração: Resuma em uma frase o módulo A7: Model Context Protocol (foc ...
  resultado: MCP padroniza como agentes acessam tools e dados externos. 

  [host] o servidor perguntou quais pastas pode acessar
  resultado: Raízes autorizadas pelo cliente: file:///content/mcp_clientes_aula


Repare na inversão: as linhas `[host]` são requisições que **vieram do servidor**.

E note a consequência de governança: o servidor produziu um resumo sem nunca ver sua chave de API, sem escolher o modelo e sem saber quanto custou. O host manteve o controle das três coisas que é exatamente o argumento de segurança do sampling.


---

# 10. O cliente dentro de um agente

Agora juntamos tudo. O loop é o mesmo ReAct do notebook de tools:

```text
pergunta -> o modelo decide -> o cliente chama -> observação -> resposta
```

A única diferença é que a ação passa por um cliente MCP, e a tool pode estar em qualquer servidor do catálogo.

Para não depender de chave de API, simulamos a decisão do modelo com regras.


In [14]:
def decidir_acao(pergunta: str) -> dict:
    """No lugar disto, um LLM receberia o catálogo de tools e escolheria."""
    texto = pergunta.lower()

    if "plano" in texto or "tempo" in texto:
        return {"tool": "curso_calcular_plano_estudo",
                "arguments": {"codigo_modulo": "A7", "minutos_disponiveis": 120}}

    if "arquivo" in texto or "resumo" in texto:
        return {"tool": "filesystem_read_text_file",
                "arguments": {"path": str(PASTA / "resumo.txt")}}

    return {"tool": "curso_consultar_modulo",
            "arguments": {"codigo_modulo": "A7"}}


async def agente_mcp(pergunta: str) -> dict:
    acao = decidir_acao(pergunta)

    async with cliente_multiplexado() as client:
        observacao = await client.call_tool(acao["tool"], acao["arguments"])

    dados = observacao.structured_content or observacao.content[0].text

    return {
        "pergunta": pergunta,
        "servidor": acao["tool"].split("_", 1)[0],
        "acao": acao,
        "observacao": dados,
        "resposta": f"Com base na observação do servidor MCP: {dados}",
    }


print_json(await agente_mcp("Me monta um plano de estudo para o módulo A7"))
print_json(await agente_mcp("O que diz o arquivo de resumo?"))


[08/01/26 15:48:02] INFO     Proxy detected connected client - reusing existing session for all        ]8;id=763787;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=676932;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py#837\837]8;;\
                             requests. This may cause context mixing in concurrent scenarios.                      

[08/01/26 15:48:03] INFO     Proxy detected connected client - reusing existing session for all        ]8;id=835670;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=768593;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py#837\837]8;;\
                             requests. This may cause context mixing in concurrent scenarios.                      

                    INFO     Proxy detected connected client - reusing existing session for all        ]8;id=121400;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=694530;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py#837\837]8;;\
                             requests. This may cause context mixing in concurrent scenarios.                      

{
  "pergunta": "Me monta um plano de estudo para o módulo A7",
  "servidor": "curso",
  "acao": {
    "tool": "curso_calcular_plano_estudo",
    "arguments": {
      "codigo_modulo": "A7",
      "minutos_disponiveis": 120
    }
  },
  "observacao": {
    "codigo_modulo": "A7",
    "minutos_disponiveis": 120,
    "sugestao": "estudar conceitos, executar exemplos e resolver exercícios"
  },
  "resposta": "Com base na observação do servidor MCP: {'codigo_modulo': 'A7', 'minutos_disponiveis': 120, 'sugestao': 'estudar conceitos, executar exemplos e resolver exercícios'}"
}


[08/01/26 15:48:07] INFO     Proxy detected connected client - reusing existing session for all        ]8;id=879585;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=109425;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py#837\837]8;;\
                             requests. This may cause context mixing in concurrent scenarios.                      

[08/01/26 15:48:08] INFO     Proxy detected connected client - reusing existing session for all        ]8;id=677542;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=327805;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py#837\837]8;;\
                             requests. This may cause context mixing in concurrent scenarios.                      

                    INFO     Proxy detected connected client - reusing existing session for all        ]8;id=373630;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=996534;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py#837\837]8;;\
                             requests. This may cause context mixing in concurrent scenarios.                      

{
  "pergunta": "O que diz o arquivo de resumo?",
  "servidor": "filesystem",
  "acao": {
    "tool": "filesystem_read_text_file",
    "arguments": {
      "path": "/content/mcp_clientes_aula/resumo.txt"
    }
  },
  "observacao": {
    "content": "MCP: um contrato comum entre agentes e sistemas.\n"
  },
  "resposta": "Com base na observação do servidor MCP: {'content': 'MCP: um contrato comum entre agentes e sistemas.\\n'}"
}


Em LangGraph, cada etapa viraria um nó: `chamar_llm`, `executar_tool_mcp`, `responder`, com uma aresta condicional decidindo se ainda há tool calls pendentes.

O cliente MCP fica dentro do nó de execução, e o grafo não precisa saber se a tool mora em um subprocesso local ou em um servidor remoto.


---

# 11. Segurança e governança do lado do cliente

O notebook anterior tratou de segurança do ponto de vista de quem expõe capacidades. Aqui somos quem **consome**, inclusive servidores de terceiros, escritos por gente que não conhecemos.

Quatro controles que pertencem ao cliente:

**Allowlist.** Nem toda tool que o servidor oferece precisa chegar ao modelo. O filesystem server, por exemplo, expõe 14 tools, incluindo escrita e movimentação de arquivos. Se sua aplicação só lê, exponha só a leitura.

**Timeout.** Um servidor que trava não pode travar o agente junto.

**Aprovação humana.** Ações destrutivas ou irreversíveis passam por confirmação, e o mesmo vale para pedidos de `sampling`.

**Desconfiança do conteúdo.** Descrições de tools e resultados vindos de servidores de terceiros entram no contexto do modelo. Um servidor malicioso pode escrever instruções ali é injeção de prompt por um canal que parece inofensivo. Trate esse texto como dado não confiável, nunca como instrução.


In [15]:
TOOLS_PERMITIDAS = {
    "curso_consultar_modulo",
    "curso_calcular_plano_estudo",
    "filesystem_read_text_file",
    "filesystem_write_file",
}

TOOLS_SENSIVEIS = {"filesystem_write_file"}


async def chamar_com_politica(
    client: Client,
    nome: str,
    argumentos: dict,
    aprovado_pelo_usuario: bool = False,
) -> dict:
    if nome not in TOOLS_PERMITIDAS:
        return {"bloqueado": True, "motivo": "fora_da_allowlist", "tool": nome}

    if nome in TOOLS_SENSIVEIS and not aprovado_pelo_usuario:
        return {"bloqueado": True, "motivo": "requer_aprovacao_humana", "tool": nome}

    resultado = await client.call_tool(nome, argumentos, timeout=10)
    return {"bloqueado": False, "tool": nome, "resultado": resultado.content[0].text}


async def demonstrar_politica() -> None:
    async with cliente_multiplexado() as client:
        print_json(await chamar_com_politica(
            client, "curso_consultar_modulo", {"codigo_modulo": "A7"}))

        print_json(await chamar_com_politica(
            client, "filesystem_write_file",
            {"path": str(PASTA / "novo.txt"), "content": "..."}))

        print_json(await chamar_com_politica(
            client, "filesystem_move_file", {"source": "a", "destination": "b"}))


await demonstrar_politica()


[08/01/26 15:48:12] INFO     Proxy detected connected client - reusing existing session for all        ]8;id=270567;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=943449;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py#837\837]8;;\
                             requests. This may cause context mixing in concurrent scenarios.                      

[08/01/26 15:48:14] INFO     Proxy detected connected client - reusing existing session for all        ]8;id=667703;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=796733;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py#837\837]8;;\
                             requests. This may cause context mixing in concurrent scenarios.                      

                    INFO     Proxy detected connected client - reusing existing session for all        ]8;id=702138;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=756647;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/providers/proxy.py#837\837]8;;\
                             requests. This may cause context mixing in concurrent scenarios.                      

{
  "bloqueado": false,
  "tool": "curso_consultar_modulo",
  "resultado": "{\n  \"codigo_modulo\": \"A7\",\n  \"titulo\": \"Model Context Protocol\",\n  \"foco\": \"padronização de integrações\",\n  \"tipo\": \"arquitetural\"\n}"
}
{
  "bloqueado": true,
  "motivo": "requer_aprovacao_humana",
  "tool": "filesystem_write_file"
}
{
  "bloqueado": true,
  "motivo": "fora_da_allowlist",
  "tool": "filesystem_move_file"
}


Note onde a política mora: **no cliente, antes de a requisição existir**. Não adianta esperar o servidor recusar o servidor de terceiros não conhece as suas regras.

É a mesma ideia da configuração da seção 2: `args` limita o que o servidor enxerga, a allowlist limita o que o modelo alcança, e a aprovação humana limita o que acontece sem alguém olhando.


---

# 12. Exercícios de fixação

## Exercício 1 — Um servidor novo

Adicione um quarto servidor à `CONFIG_MCP` e liste suas tools.

Sugestões de servidores oficiais que não precisam de chave:

```json
"git":   { "command": "uvx", "args": ["mcp-server-git", "--repository", "."] },
"fetch": { "command": "uvx", "args": ["mcp-server-fetch"] }
```

Se o servidor falhar ao iniciar, investigue: servidores Python de referência costumam precisar de uma versão específica do SDK, e `uvx --with 'mcp==1.12.0' ...` resolve o conflito. Diagnosticar isso é parte do trabalho de quem opera clientes.


In [16]:
# Escreva sua solução aqui.


## Exercício 2 — Descrição automática de um servidor

Escreva `async def descrever_servidor(session) -> dict` que devolva um resumo com:

- nome e versão do servidor;
- capacidades declaradas;
- quantidade de tools, resources e prompts;
- nomes das tools que declaram `outputSchema`.

Teste contra o servidor do curso e contra o DeepWiki, e compare.


In [17]:
# Escreva sua solução aqui.


## Exercício 3 — Cliente resiliente

Modifique `agente_mcp` para tratar os dois tipos de erro:

- se `isError` for verdadeiro, reinjete a mensagem na "decisão" e tente uma segunda vez com outro argumento;
- se vier exceção, registre e devolva uma resposta explicando que a integração falhou.

Teste pedindo um módulo que não existe.


In [18]:
# Escreva sua solução aqui.


## Exercício 4 — Sampling com LLM de verdade

Troque o `tratar_sampling` simulado por uma chamada real ao provedor que você usa.

Antes de gerar, imprima o pedido e peça confirmação com `input()`. Se o usuário recusar, levante um erro em vez de devolver texto — a especificação reserva o código `-1` para "usuário recusou a requisição de sampling".


In [19]:
# Escreva sua solução aqui.


## Exercício 5 — Allowlist por servidor

Generalize a política da seção 11: em vez de uma lista fixa de nomes, escreva uma configuração por servidor, do tipo

```python
POLITICA = {
    "curso": "todas",
    "filesystem": {"read_text_file", "list_directory"},
    "deepwiki": "nenhuma",
}
```

e uma função que filtre o catálogo de `list_tools()` antes de entregá-lo ao modelo.


In [20]:
# Escreva sua solução aqui.


## Exercício 6 — Comparando transportes

Meça o tempo de `initialize()` + `list_tools()` nos três transportes que usamos: em memória, stdio e HTTP.

Explique a diferença. O que exatamente acontece em cada caso antes da primeira resposta chegar?


In [21]:
# Escreva sua solução aqui.


---

# 13. Resumo da aula

Neste notebook, escrevemos clientes MCP contra servidores.

Os pontos principais:

- um cliente cuida de transporte, handshake, correlação de mensagens, descoberta e tradução de erros;
- a relação cliente-servidor é 1:1; um host com N servidores mantém N clientes;
- o arquivo `mcpServers` é a forma padrão de declarar servidores, e cada entrada vira um cliente;
- `command`/`args` descrevem um servidor local; `url`, um remoto;
- `env` é onde entram os segredos e `args` é onde se limita o escopo;
- a descoberta via `inputSchema`/`outputSchema` permite usar um servidor sem documentação;
- `isError` chega como resultado e vira contexto para o modelo; erro de protocolo chega como exceção;
- o transporte em memória serve para testar seu próprio servidor sem subir processo;
- Streamable HTTP conecta a servidores remotos sem mudar o código de aplicação;
- um cliente multiplexado agrega vários servidores e roteia pelo prefixo `servidor_tool`;
- o protocolo é bidirecional: `sampling`, `roots` e `elicitation` são requisições que o servidor faz ao cliente;
- allowlist, timeout, aprovação humana e desconfiança do conteúdo são responsabilidades do cliente.


## 13.1 Checklist de compreensão

1. Quais são as cinco responsabilidades de um cliente MCP?
2. Por que um host com três servidores precisa de três clientes?
3. O que muda no código ao trocar stdio por HTTP? E o que não muda?
4. Como um cliente descobre como chamar uma tool que nunca viu?
5. Qual a diferença, do lado do cliente, entre `isError=True` e uma exceção `McpError`?
6. Quando o transporte em memória é a escolha certa?
7. Por que as tools vêm prefixadas no cliente multiplexado?
8. O que é sampling e por que ele preserva o controle do host sobre modelo e custo?
9. Para que serve `roots`?
10. Por que a allowlist precisa estar no cliente e não no servidor?
11. Como um servidor de terceiros poderia tentar injetar instruções no seu modelo?


## 13.2 Próximos passos

Agora que você sabe escrever clientes MCP, os próximos passos naturais são:

- integrar um cliente MCP como fonte de tools dentro de um agente em LangGraph, substituindo `decidir_acao` por um nó real de chamada ao LLM (ver seção 10);
- combinar múltiplos servidores multiplexados (seção 8) com a camada de memória vista nos notebooks anteriores, para que o agente lembre de decisões tomadas com cada servidor;
- explorar autenticação OAuth para servidores remotos privados (mencionada na seção 7), quando o servidor não for público como o DeepWiki.


---

# 14. Referências

- Model Context Protocol Arquitetura: https://modelcontextprotocol.io/specification/2025-11-25/architecture
- Model Context Protocol Ciclo de vida e inicialização: https://modelcontextprotocol.io/specification/2025-11-25/basic/lifecycle
- Model Context Protocol Transports: https://modelcontextprotocol.io/specification/2025-11-25/basic/transports
- Model Context Protocol Tools: https://modelcontextprotocol.io/specification/2025-11-25/server/tools
- Model Context Protocol Resources: https://modelcontextprotocol.io/specification/2025-11-25/server/resources
- Model Context Protocol Sampling: https://modelcontextprotocol.io/specification/2025-11-25/client/sampling
- Model Context Protocol Roots: https://modelcontextprotocol.io/specification/2025-11-25/client/roots
- Model Context Protocol Elicitation: https://modelcontextprotocol.io/specification/2025-11-25/client/elicitation
- SDK oficial em Python: https://github.com/modelcontextprotocol/python-sdk
- Servidores de referência: https://github.com/modelcontextprotocol/servers
- FastMCP: https://gofastmcp.com
- Notebook anterior: N5 Model Context Protocol (MCP)

> Nota sobre versões: os links acima apontam para a revisão `2025-11-25`, que é a implementada pelos SDKs usados aqui. A especificação já publicou revisões mais novas verifique qual versão seu SDK negocia antes de usar recursos recentes.
